# Cifar LTM pre-training
This notebook is used to pre-train the LTM on a specific combination of coarse and fine classes.
The LTM state dict will be serialized so it can be loaded and used for fine-tuning or with an STM.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from dataclasses import dataclass

import torch
import torch.nn.functional as F
import torchvision.utils as vutils
from model.resnet import ResNetConfig
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from environment.cifar.cifar_classifier import CifarClassifier
from environment.cifar.cifar_dataset import Cifar100Dataset
from util.log import get_run_path

run_root_path = "cifar_100_pretrain"
run_path = get_run_path(
    prefix = run_root_path, 
    path = "./runs",
)
data_file_path = "/home/dave/dev/cifar-100-python"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_epochs = 40
batch_size = 128
learning_rate = 1e-3
model_name = "cifar_100_subclasses_12"


In [ ]:

# Train on fine sub-classes 1, 2 
training_exclude_classes_fine = Cifar100Dataset.get_fine_classes([    3,4,5,])

# Eval on fine sub-classes 3, 4, 5
#evaluate_exclude_classes_fine = Cifar100Dataset.get_fine_classes([1,2,      ])
evaluate_exclude_classes_fine = training_exclude_classes_fine

In [ ]:
dataset_training = Cifar100Dataset(
    file_path=data_file_path, 
    label_type=Cifar100Dataset.LABEL_TYPE_COARSE,
    training=True,
    exclude_classes_coarse=None,
    exclude_classes_fine=training_exclude_classes_fine,
    as_tensor=True,
)

dataset_evaluate = Cifar100Dataset(
    file_path=data_file_path, 
    label_type=Cifar100Dataset.LABEL_TYPE_COARSE,
    training=False,
    exclude_classes_coarse=None,
    exclude_classes_fine=evaluate_exclude_classes_fine,
    as_tensor=True,
)

loader_training = DataLoader(
    dataset_training,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

loader_evaluate = DataLoader(
    dataset_evaluate,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [ ]:
config = ResNetConfig(
    num_classes=dataset_training.get_num_classes(),
)
model = CifarClassifier(config, bias_stage=-1).to(device)
num_parameters = sum(p.numel() for p in model.parameters())
print(f"# parameters:{num_parameters}")

In [ ]:
debug_images = False
if debug_images:
    images, labels = next(iter(loader_training))

    # Create grids out of the batches to see multiple samples at once
    # normalize=True ensures the values are scaled properly between 0 and 1 for display
    grid = vutils.make_grid(images, nrow=4, normalize=True)

    # Log to TensorBoard
    # The global_step parameter lets you track how augmentations change over epochs if called inside the loop
    writer = SummaryWriter(log_dir='./runs/cifar_images')
    writer.add_image('Cifar', grid, global_step=0)
    writer.close()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
writer = SummaryWriter(log_dir=run_path)

In [ ]:
@dataclass
class EpochMetrics:
    mean_loss:float = 0
    mean_accuracy:float = 0
    num_samples:int = 0
    global_step:int = 0

def do_epoch_mode(
    model,
    loader,
    optimizer,
    device,
    training:bool,
    global_step:int,
    max_steps:int = 0,
    log_period:int = 100,
) -> EpochMetrics:
    if training:
        model.train()
        epoch_type = "Training"
    else:
        model.eval()
        epoch_type = "Evaluate"
        
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_samples = 0

    log_loss = 0.0
    log_correct = 0
    log_samples = 0
    num_steps = 0

    for x, y in loader:

        if max_steps > 0 and num_steps >= max_steps:
            break  # early truncation of epoch

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad()

            logits, encoding = model(x, bias=None)

            loss = F.cross_entropy(
                logits,
                y,
            )

            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                logits, encoding = model(x, bias=None)

                loss = F.cross_entropy(
                    logits,
                    y,
                )

        samples_step = x.size(0)
        loss_step  = (
            loss.item() * samples_step
        )
        epoch_loss += loss_step

        correct_step = (
            (logits.argmax(dim=1) == y)
            .sum()
            .item()
        )
        epoch_correct += correct_step
        epoch_samples += samples_step

        log_loss += loss_step
        log_correct += correct_step
        log_samples += samples_step

        if (global_step < 100) or ((log_samples % log_period) == 0):
            log_accuracy = log_correct / log_samples
            writer.add_scalar(f'{epoch_type} Loss {log_period}', loss_step, global_step)
            writer.add_scalar(f'{epoch_type} Accuracy {log_period}', log_accuracy, global_step)
            log_loss = 0.0
            log_correct = 0
            log_samples = 0

        global_step += 1
        num_steps += 1

    epoch_metrics = EpochMetrics(
        mean_loss = epoch_loss / epoch_samples,
        mean_accuracy =  epoch_correct / epoch_samples,
        num_samples = epoch_samples,
        global_step = global_step,
    )
    return epoch_metrics

In [ ]:
def do_epoch(
    epoch:int,
    global_step_training:int,
    global_step_evaluate:int,
):

    metrics_training = do_epoch_mode(
        model,
        loader_training,
        optimizer,
        device,
        training = True,
        global_step = global_step_training,
        max_steps = 0,
    )
    global_step_training += metrics_training.global_step

    metrics_evaluate = do_epoch_mode(
        model,
        loader_evaluate,
        optimizer,
        device,
        training = False,
        global_step = global_step_evaluate,
        max_steps = 0,
    )
    global_step_evaluate += metrics_evaluate.global_step

    writer.add_scalar('Training-Loss', metrics_training.mean_loss, global_step_training)
    writer.add_scalar('Training-Accuracy', metrics_training.mean_accuracy, global_step_training)

    writer.add_scalar('Evaluate-Loss', metrics_evaluate.mean_loss, global_step_evaluate)
    writer.add_scalar('Evaluate-Accuracy', metrics_evaluate.mean_accuracy, global_step_evaluate)

    print(
        f"Epoch {epoch + 1:3d}/{num_epochs} "
        f"| train loss {metrics_training.mean_loss:.4f} "
        f"| train acc {metrics_training.mean_accuracy:.3f} "
        f"| test loss {metrics_evaluate.mean_loss:.4f} "
        f"| test acc {metrics_evaluate.mean_accuracy:.3f}"
    )
    return global_step_training, global_step_evaluate


In [ ]:
global_step_training = 0
global_step_evaluate = 0


In [ ]:
for epoch in range(num_epochs):
    global_step_training, global_step_evaluate = do_epoch(epoch, global_step_training, global_step_evaluate)


In [ ]:
        
# Save the frozen encoder asset for downstream PPO modulation experiments
model_dir = run_root_path
os.makedirs(f'./{model_dir}', exist_ok=True)
torch.save(model.state_dict(), f'./{model_dir}/{model_name}_e{num_epochs}.pth')
writer.close()
print("Model saved successfully.")
